In [1]:
from gpaw.new.ase_interface import GPAW
from ase import Atoms
import numpy as np
from gpaw import FermiDirac
from ase.visualize import view
import ase

## Primitive cell

In [14]:
primitive = ase.io.read("1MnI2-1.cif")

In [15]:
primitive

Atoms(symbols='MnI2', pbc=True, cell=[[4.165715878531158, 0.0, 0.0], [-2.082857939265579, 3.6076157757561913, 0.0], [0.0, 0.0, 18.02743121316087]], spacegroup_kinds=...)

## Create magnetic supercell

I think the above would not give the same bands.

Note that the supercell is different from the one in the paper.
In the paper they use $Q = (1/3,1/3)$

I think the above corresponds to $Q=(1/3,0)$

IT is the same local 120 degree spin structure. but it is not the same supercell, so the bands might differ.

Trying to create the one they use.

In [16]:
#Define transformation matrix from primitive to magnetic cell
P = np.array([
    [2, 1, 0],
    [-1, 1, 0],
    [0, 0, 1]
])

#defines supercell basis as linear combination of primitive basis
#Unsure if this is defined correctly.

# So Supercell A1 = 2a_1+1a_2
#A2=-a1+a2

Corrected to different supercell above now (initial one didn't converge anyway)

In [17]:
from ase.build import make_supercell

supercell = make_supercell(primitive, P)

## Creating spiral magnetic moments

The magnetization density satisfies:
$$\mathbf{m(r+r_i)} = R_{\hat n}(\mathbf{q\cdot r_i})\mathbf{m(r)}$$
(see the paper).
Take the reference spin $m_0=(0,m,0)$. Then:

$$
\mathbf{m(r+r_i)} = \begin{pmatrix} \cos(\varphi) & -\sin(\varphi)\\ \sin(\varphi) & \cos(\varphi) \end{pmatrix}
\begin{pmatrix} 0 \\ m \end{pmatrix} = \begin{pmatrix} -m\sin(\varphi) \\ m\cos(\varphi) \end{pmatrix}
$$
With $\varphi = \mathbf{q\cdot r_i}$

So, we can calculate the other spins easily based on a reference spin in the y-direction. The spiral does not rotate in the z-direction, thus the 2x2 above.

We could have chosen the reference spin to point in any direction. This physically changes nothing. I chose the y-direction only so the spins align exactly in the way we see them in Fig 1 of the paper.

**Important note about chirality:**

Currently we have a counter-clockwise rotating spiral in the above. If I read the paper Fig 1 d correctly, they have a clockwise rotating spiral. This corresponds to changing $\mathbf{Q} \rightarrow \mathbf{-Q}$.

When there is no SOC (or other symmetry breaking terms), these two options should be identical. SOC breaks the symmetry though, and the chiralities might yield different results.

Below, I include two options. The first is reference spin in x, counterclockwise rotation.

If we wish to Match Fig 1d visually completely, then I found that a reference spin in the y-direction and clock-wise rotating spiral (see -Q in the code) will do the trick.


**Choosing reference spin in the x-direction, counterclockwise rotating spiral**

In [38]:
# m = 4.5
# magmoms = np.zeros((len(supercell), 3))


# A = primitive.cell[:2, :2] #extracts the 2D lattice vectors from the PRIMITIVE CELL. It is a 2x2 matrix.
# A_inv = np.linalg.inv(A.T) #Invert the above, constructs a mapping from cartesian to lattice coordinates. Transposed since ASE stores lattice vectors as rows, we need them as columns.
#                             #This gives \vec{n}=A^{-1}\vec{r}, where r is cartesian position, n is coordinates in lattice basis
# Q = np.array([1/3, 1/3])    #Defines the magnetic ordering vector in reciprocal lattice coordinates.

# for i, atom in enumerate(supercell):
#     if atom.symbol != 'Mn':
#         continue

#     r_cart = atom.position[:2] #extracts the cartesian position of the Mn atom
#     n = A_inv @ r_cart         # Converts to lattice coordinates \vec{r}=n_1a_1+n_2a_2

#     phase = 2 * np.pi * np.dot(Q, n) #each lattice site gets a phase angle depending on its position (given via the lattice coordinates n)

#     magmoms[i] = [                   #Assigns a planar spin spiral
#         m * np.cos(phase),
#         m * np.sin(phase),
#         0.0
#     ]

# supercell.set_initial_magnetic_moments(magmoms)

In [39]:
# view(supercell)
# #In the GUI, go to View->show Magmoms. Also, View->repeat, 2 in x and y

**Choosing reference spin in the y-direction, clockwise rotating spiral**

Matches Fig 1d visually exactly.

In [20]:
m = 4.5
magmoms = np.zeros((len(supercell), 3))


A = primitive.cell[:2, :2] #extracts the 2D lattice vectors from the PRIMITIVE CELL. It is a 2x2 matrix.
A_inv = np.linalg.inv(A.T) #Invert the above, constructs a mapping from cartesian to lattice coordinates. Transposed since ASE stores lattice vectors as rows, we need them as columns.
                            #This gives \vec{n}=A^{-1}\vec{r}, where r is cartesian position, n is coordinates in lattice basis
Q = np.array([1/3, 1/3])    #Defines the magnetic ordering vector in reciprocal lattice coordinates.

for i, atom in enumerate(supercell):
    if atom.symbol != 'Mn':
        continue

    r_cart = atom.position[:2] #extracts the cartesian position of the Mn atom
    n = A_inv @ r_cart         # Converts to lattice coordinates \vec{r}=n_1a_1+n_2a_2

    phase = 2 * np.pi * np.dot(-Q, n) #each lattice site gets a phase angle depending on its position (given via the lattice coordinates n)
    #NOTE minus Q above for clockwise

    magmoms[i] = [                   #Assigns a planar spin spiral, note that m_0=(0,m,0) to get this.
        -m * np.sin(phase),
        m * np.cos(phase),
        0.0
    ]
supercell.set_initial_magnetic_moments(magmoms)

In [ ]:
view(supercell)
# #In the GUI, go to View->show Magmoms. Also, View->repeat, 2 in x and y

<Popen: returncode: None args: ['/home/runerce/Desktop/DFT-project-2026/.ven...>

In [22]:
supercell.arrays

{'numbers': array([25, 53, 53, 25, 53, 53, 25, 53, 53]),
 'positions': array([[-2.99569531e-18,  5.17647340e-18,  9.01371561e+00],
        [ 2.08285794e+00,  1.20253859e+00,  1.06632503e+01],
        [-1.76049209e-16,  2.40507718e+00,  7.36418087e+00],
        [-2.08285794e+00,  3.60761578e+00,  9.01371561e+00],
        [-2.23884421e-15,  4.81015437e+00,  1.06632503e+01],
        [ 4.16571588e+00,  2.40507718e+00,  7.36418087e+00],
        [ 2.08285794e+00,  3.60761578e+00,  9.01371561e+00],
        [-2.08285794e+00,  1.20253859e+00,  1.06632503e+01],
        [-4.16571588e+00,  2.40507718e+00,  7.36418087e+00]]),
 'spacegroup_kinds': array([0, 1, 2, 0, 1, 2, 0, 1, 2]),
 'initial_magmoms': array([[ 1.35074009e-17,  4.50000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 3.89711432e+00, -2.25000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00]

## GPAW calculator and SCF calc

In [ ]:
# --- 4. Setup GPAW Calculator ---
# Article specifies: LDA functional, 600 eV cutoff.
# Symmetry must be off for non-collinear spirals.
# k-points: The magnetic BZ is smaller. If primitive was 12x12x1, supercell needs 4x12x1?
# Let's use a dense grid to resolve the bands smoothly for Fig 1.
calc = GPAW(
    mode={'name':'pw',
          'ecut':300},          # 600 eV cutoff in per paper. Rough first guess
    xc='LDA',              # Paper explicitly uses LDA 
    mixer={'backend': 'pulay',              #This was used to mimic https://gpaw.readthedocs.io/tutorialsexercises/magnetic/spinspiral/spinspiral.html#spin-spiral-calculations
                       'beta': 0.05,
                       'method': 'sum',
                       'nmaxold': 5,
                       'weight': 100},
    kpts={'size':(3,3,1), 'gamma':True},       # Adjusted for the rhombus supercell
    symmetry='off',        # Crucial for spiral
    magmoms=magmoms,       # Enforce non-collinear start
    spinpol=True,          # Needed for non-collinear
    occupations=FermiDirac(0.01),
    txt='MnI2_spiral_supercell_attempt2.txt',
    parallel={'domain': 1, 'band': 1},
    maxiter=50,
)

calc.verbosity=1

supercell.calc = calc

### Run SCF calc

In [ ]:
# --- 5. Run Calculation ---
print("Running SCF for magnetic supercell...")
energy = supercell.get_potential_energy()
calc.write('MnI2_spiral_gs_attempt2.gpw')

Running SCF for magnetic supercell...


In [102]:
#Checking convergence and magnetic moments
energy = supercell.get_potential_energy()
print('energy=',energy) #check for divergence. If it diverges, something wrong in the setup
print('')
#Magmoms
print('Magnetic moments = ', supercell.get_magnetic_moments())
#We want three Mn moments, roughly 120 degrees apart, magnitude around 3-5 µB
print('')
#Total M
print('Total magnetic moment=', supercell.get_magnetic_moment())
#Should be 0. If they align ferromagnetically, phase may be incorrectly assigned

#Encountering some strangeness. Printing all arrays.
# print('All arrays')
# print(supercell.arrays)

energy= -43.19182025125478

Magnetic moments =  [-1.97139021e-14  6.61761749e-12 -2.85507220e-15  9.99952153e-14
  1.06331324e-15  6.61589651e-12 -8.34002176e-14 -6.61615166e-12
 -6.61438036e-12]

Total magnetic moment= -1.5197580387955155e-15


In [ ]:
# THIS SHOWS THE LOCAL MAGNETIC MOMENTS AFTER THE CALCULATION
calc.get_non_collinear_magnetic_moments()

array([[ 2.47728431e-14,  4.20399224e+00, -1.97139021e-14],
       [ 1.10337189e-06,  1.31850560e-05,  6.61761749e-12],
       [ 8.98492233e-15,  1.12731814e-05, -2.85507220e-15],
       [ 3.64129207e+00, -2.10106725e+00,  9.99952153e-14],
       [-1.23054330e-14,  1.12731900e-05,  1.06331324e-15],
       [-1.10332552e-06,  1.31850651e-05,  6.61589651e-12],
       [-3.64129207e+00, -2.10106725e+00, -8.34002176e-14],
       [-1.10337188e-06,  1.31850560e-05, -6.61615166e-12],
       [ 1.10332550e-06,  1.31850651e-05, -6.61438036e-12]])

## Loading the GPW file
Instead of running the SCF code every time

In [ ]:
calc2 = GPAW('MnI2_spiral_gs_attempt2.gpw')
atoms = calc2.get_atoms()

In [ ]:
# view(atoms)

<Popen: returncode: None args: ['/home/runerce/Desktop/DFT-project-2026/.ven...>

As seen above, the "Atoms" object isn't updated with the resulting spin structure from the GPAW calculation.

There is an issue with the ASE GUI when there is a calculator attached to the object. The stuff below is just to view the resulting structure after the calculation

In [33]:
atoms2 = calc2.get_atoms()
print('local moments after calc')
print(calc2.get_non_collinear_magnetic_moments()) #Show the calculated local moments)

atoms2.arrays['magmoms'] = calc2.get_non_collinear_magnetic_moments() #Forcing the Atoms magmoms to equal the calculated ones
atoms2.calc = None

local moments after calc
[[ 2.47728431e-14  4.20399224e+00 -1.97139021e-14]
 [ 1.10337189e-06  1.31850560e-05  6.61761749e-12]
 [ 8.98492233e-15  1.12731814e-05 -2.85507220e-15]
 [ 3.64129207e+00 -2.10106725e+00  9.99952153e-14]
 [-1.23054330e-14  1.12731900e-05  1.06331324e-15]
 [-1.10332552e-06  1.31850651e-05  6.61589651e-12]
 [-3.64129207e+00 -2.10106725e+00 -8.34002176e-14]
 [-1.10337188e-06  1.31850560e-05 -6.61615166e-12]
 [ 1.10332550e-06  1.31850651e-05 -6.61438036e-12]]


In [ ]:
# view(atoms2)

<Popen: returncode: None args: ['/home/runerce/Desktop/DFT-project-2026/.ven...>

The above visually shows that our spin spiral structure is still present

## Bandstructure

We are working in the magnetic supercell, so I guess(?) we will compute bands in the magnetic Brillouin Zone.

In [35]:
from ase.dft.kpoints import bandpath

In [ ]:
M = [0.5, 0.0, 0.0]
minus_M = [-0.5, 0.0, 0.0]
Gamma = [0.0, 0.0, 0.0]

kpts = np.array([
    minus_M,
    Gamma,
    M
])

path = bandpath(kpts, supercell.cell, npoints=50)

In [37]:
#Non SCF band calc
calc = GPAW('MnI2_spiral_gs.gpw').fixed_density(
    kpts=path,
    symmetry='off'
)


Diagonalizing LCAO Hamiltonian
Converting LCAO to grid
convergence criteria:
- Maximum integral of absolute [eigenst]ate change: 4e-08 eV^2 / valence electron
maximum number of iterations: 50



KeyboardInterrupt: 

In [ ]:
bs = calc.band_structure()

In [ ]:
import matplotlib.pyplot as plt

bs.plot(emin=-3, emax=3)
plt.show()